# Dataset audit and immutable manifest

Objective: build a checksum-backed manifest, surface corrupt or unresolved images, and detect content or grouping leakage across splits before model training.

Success means the manifest verifies cryptographically and every configured quality gate passes. A failed gate is a dataset finding, not a notebook failure.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').is_file()
)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from deepfake_detection.data import (  # noqa: E402
    audit_manifest,
    build_manifest,
    load_dataset_spec,
    verify_manifest,
    write_audit_report,
)

CONFIG_PATH = PROJECT_ROOT / 'configs/datasets/kaggle_2026.yaml'
spec = load_dataset_spec(CONFIG_PATH)
spec.name, spec.version

## Plan

1. Confirm the configured source files are present.
2. Build or reuse the content-addressed manifest.
3. Verify its checksums.
4. Review status, label, split, duplicate, group-overlap, and metadata-leakage findings.

In [ ]:
readiness = {
    'data_root': str(spec.data_root),
    'data_root_exists': spec.data_root.is_dir(),
    'metadata_path': str(spec.metadata_path),
    'metadata_exists': spec.metadata_path.is_file(),
}
ready = readiness['data_root_exists'] and readiness['metadata_exists']
readiness

In [ ]:
artifact = None
report = None
audit_artifact = None

if ready:
    artifact = build_manifest(spec)
    verified, verification_errors = verify_manifest(artifact.directory)
    report = audit_manifest(artifact.manifest_path, spec)
    audit_artifact = write_audit_report(report, spec)
    run_summary = {
        'manifest_id': artifact.manifest_id,
        'rows': artifact.row_count,
        'manifest_reused': artifact.reused_existing,
        'checksum_verified': verified,
        'verification_errors': verification_errors,
        'audit_passed': audit_artifact.passed,
        'audit_report': str(audit_artifact.report_path),
    }
else:
    run_summary = {
        'skipped': True,
        'reason': 'Place metadata and images at the configured paths, then rerun all cells.',
    }
run_summary

## Results

Quality-gate failures identify work required before training. Warnings call out suspicious metadata relationships that should not be used as input features.

In [ ]:
if report is not None:
    findings = {
        'summary': report['summary'],
        'failures': report['quality_gates']['failures'],
        'warnings': report['quality_gates']['warnings'],
        'group_overlap': report['group_overlap'],
        'metadata_leakage_indicators': report['metadata_leakage_indicators'],
    }
    print(json.dumps(findings, indent=2))
else:
    print('No audit report yet.')

In [ ]:
if report is not None:
    figure, axes = plt.subplots(1, 3, figsize=(13, 3.5))
    for axis, key, title in zip(
        axes,
        ('status_counts', 'label_counts', 'split_counts'),
        ('Decode/path status', 'Labels', 'Splits'),
        strict=True,
    ):
        pd.Series(report['summary'][key]).plot.bar(ax=axis, title=title)
        axis.tick_params(axis='x', rotation=35)
    figure.tight_layout()
else:
    print('Charts will appear after the dataset is available.')

## Next steps

- Resolve every missing, ambiguous, unsafe, or corrupt sample.
- Remove cross-split duplicates and keep identities/videos within one split.
- Treat high-purity metadata fields as evaluation slices only.
- Freeze the passing manifest ID in the training configuration.